# Maia Corpus v9 — Auditoria Gutenberg v0.7
**Auditoria somente-leitura dos livros integrais adquiridos pelo v0.6**

Confere integridade SHA-256, bytes e caracteres; identifica IDs repetidos, arquivos órfãos e divergências; produz estatísticas por idioma, indícios de cabeçalhos/rodapés do Gutenberg e inventário para revisão de direitos autorais.

**Não modifica os arquivos v0.6**, não remove cabeçalhos e **não aprova** livros para treinamento. Salva relatórios e inventários em `gutenberg_v07_audit/` no Google Drive. Não instala bibliotecas e não acessa a internet.

## 1. Montar Drive e localizar corpus v0.6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, csv, hashlib, re, statistics, datetime, collections
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
SRC=ROOT/'gutenberg_v06'
OUT=ROOT/'gutenberg_v07_audit'
OUT.mkdir(parents=True,exist_ok=True)
assert (SRC/'manifests').is_dir(), f'Manifestos v0.6 não encontrados: {SRC}'
LANGUAGES=['pt','en','es']
print('Origem (somente leitura):',SRC)
print('Destino:',OUT)

## 2. Configuração e critérios
- `saved` é a única situação que corresponde a documento persistido.
- Confere SHA-256 **dos bytes** e compara o tamanho com `bytes` do manifesto.
- Contagem de caracteres usa UTF-8 estrito; arquivos inválidos são sinalizados.
- Detecta marcadores Gutenberg nas primeiras/últimas 8.000 posições, sem alterar conteúdo.
- O campo de direitos permanece `pending_manual_review` até conferência por obra, edição e jurisdição (Brasil e outras relevantes).

In [ ]:
MARKERS_START=[
    r'\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG',
    r'START OF THE PROJECT GUTENBERG',
    r'PROJECT GUTENBERG EBOOK',
    r'PROJETO GUTENBERG',
]
MARKERS_END=[
    r'\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG',
    r'END OF THE PROJECT GUTENBERG',
    r'END OF PROJECT GUTENBERG',
]
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('Auditoria:',STAMP)

## 3. Ler manifestos e verificar integridade (pode demorar no Google Drive)

In [ ]:
def file_sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()

rows=[]
manifest_issues=[]
for lang in LANGUAGES:
    path=SRC/'manifests'/f'{lang}.jsonl'
    if not path.exists():
        manifest_issues.append({'language':lang,'issue':'missing_manifest','path':str(path)})
        continue
    for line_number,line in enumerate(path.open(encoding='utf-8'),1):
        try:
            item=json.loads(line)
            if not isinstance(item,dict): raise ValueError('Not an object')
        except (ValueError,UnicodeError) as exc:
            manifest_issues.append({'language':lang,'issue':'invalid_json',
                                    'line':line_number,'detail':str(exc)})
            continue
        if item.get('status')!='saved':
            continue
        sha=item.get('sha256')
        if not isinstance(sha,str) or not re.fullmatch(r'[a-f0-9]{64}',sha):
            manifest_issues.append({'language':lang,'issue':'invalid_sha','line':line_number})
            continue
        # Never trust manifest absolute paths; derive from known root.
        bookpath=SRC/'quarantine'/'books'/lang/(sha+'.txt')
        row={'language':lang,'book_id':str(item.get('book_id') or ''),
             'sha256':sha,'manifest_line':line_number,
             'manifest_bytes':item.get('bytes'),'manifest_characters':item.get('characters'),
             'source_dataset':item.get('source_dataset',''),
             'path':str(bookpath),'exists':bookpath.is_file(),
             'sha_ok':False,'bytes_ok':None,'characters_ok':None,
             'actual_bytes':None,'actual_characters':None,'utf8_ok':False,
             'gutenberg_start_marker':False,'gutenberg_end_marker':False,
             'gutenberg_any_marker':False,'possible_header_footer':False,
             'review_status':'pending_manual_review',
             'copyright_brazil':'not_verified',
             'copyright_source':'not_verified'}
        if row['exists']:
            try:
                data=bookpath.read_bytes()
                row['actual_bytes']=len(data)
                row['sha_ok']=hashlib.sha256(data).hexdigest()==sha
                if isinstance(item.get('bytes'),int):
                    row['bytes_ok']=len(data)==item['bytes']
                text=data.decode('utf-8',errors='strict')
                row['utf8_ok']=True
                row['actual_characters']=len(text)
                if isinstance(item.get('characters'),int):
                    row['characters_ok']=len(text)==item['characters']
                head=text[:8000]; tail=text[-8000:]
                row['gutenberg_start_marker']=any(re.search(x,head,re.I) for x in MARKERS_START)
                row['gutenberg_end_marker']=any(re.search(x,tail,re.I) for x in MARKERS_END)
                row['gutenberg_any_marker']=bool(re.search('project gutenberg|projeto gutenberg',head+' '+tail,re.I))
                row['possible_header_footer']=row['gutenberg_any_marker']
            except Exception as exc:
                row['read_error']=type(exc).__name__+': '+str(exc)
        rows.append(row)
    print(lang,'registros salvos indexados:',sum(r['language']==lang for r in rows),flush=True)
print('Total linhas saved:',len(rows),'problemas de manifesto:',len(manifest_issues))

## 4. IDs repetidos, arquivos órfãos e divergências

In [ ]:
id_to_hashes=collections.defaultdict(set)
sha_to_ids=collections.defaultdict(set)
for r in rows:
    if r['book_id']: id_to_hashes[(r['language'],r['book_id'])].add(r['sha256'])
    sha_to_ids[(r['language'],r['sha256'])].add(r['book_id'])
repeated_ids={k:sorted(v) for k,v in id_to_hashes.items() if len(v)>1}
duplicate_manifest_sha=collections.Counter((r['language'],r['sha256']) for r in rows)
duplicate_manifest_sha={k:v for k,v in duplicate_manifest_sha.items() if v>1}
orphans=[]
for lang in LANGUAGES:
    folder=SRC/'quarantine'/'books'/lang
    indexed={r['sha256'] for r in rows if r['language']==lang}
    if folder.is_dir():
        for path in folder.glob('*.txt'):
            if path.stem not in indexed:
                orphans.append({'language':lang,'path':str(path),'sha256_filename':path.stem})
failures=[r for r in rows if (not r['exists'] or not r['sha_ok'] or
          r['bytes_ok'] is False or r['characters_ok'] is False or not r['utf8_ok'])]
print('IDs associados a mais de um SHA:',len(repeated_ids))
print('SHA repetidos no manifesto:',len(duplicate_manifest_sha))
print('Arquivos órfãos:',len(orphans))
print('Falhas de integridade:',len(failures))
for r in failures[:5]: print('FALHA',r['language'],r['book_id'],r['sha256'][:12],r.get('read_error'))

## 5. Estatísticas por idioma e indicadores de Gutenberg

In [ ]:
stats=[]
for lang in LANGUAGES:
    group=[r for r in rows if r['language']==lang]
    valid=[r for r in group if r['sha_ok'] and r['utf8_ok']]
    sizes=[r['actual_bytes'] for r in valid]
    chars=[r['actual_characters'] for r in valid]
    stats.append({
        'language':lang,'manifest_saved_rows':len(group),
        'unique_sha':len({r['sha256'] for r in group}),
        'valid_integrity':len(valid),
        'failed_integrity':len([r for r in group if r not in valid]),
        'total_bytes':sum(sizes),'total_characters':sum(chars),
        'mean_characters':round(statistics.mean(chars),1) if chars else 0,
        'median_characters':statistics.median(chars) if chars else 0,
        'min_characters':min(chars) if chars else 0,
        'max_characters':max(chars) if chars else 0,
        'start_markers':sum(r['gutenberg_start_marker'] for r in valid),
        'end_markers':sum(r['gutenberg_end_marker'] for r in valid),
        'possible_header_footer':sum(r['possible_header_footer'] for r in valid),
    })
for s in stats: print(json.dumps(s,ensure_ascii=False,indent=2))

## 6. Inventário para revisão de direitos
O v0.6 **não guarda título, autor, ano de falecimento ou edição** no manifesto. Por isso, esses campos ficam vazios; não são inferidos a partir do ID. O inventário inclui identificador, idioma, hash, tamanho e campos para pesquisa jurídica posterior.

**Não preencha automaticamente** `approved_for_training` com verdadeiro: a decisão requer evidências e revisão.

In [ ]:
inventory=[]
seen=set()
for r in rows:
    key=(r['language'],r['sha256'])
    if key in seen: continue
    seen.add(key)
    inventory.append({
        'language':r['language'],'book_id':r['book_id'],'sha256':r['sha256'],
        'path':r['path'],'bytes':r['actual_bytes'],'characters':r['actual_characters'],
        'integrity_ok':bool(r['sha_ok'] and r['utf8_ok'] and r['bytes_ok'] is not False
                            and r['characters_ok'] is not False),
        'possible_gutenberg_boilerplate':r['possible_header_footer'],
        'title':'','author':'','author_death_year':'','edition':'',
        'original_publication_year':'','source_rights_evidence_url':'',
        'brazil_rights_evidence_url':'','rights_review_notes':'',
        'rights_status':'pending_manual_review','approved_for_training':False
    })
print('Obras/arquivos únicos no inventário:',len(inventory))

## 7. Salvar resultados no Drive (sem alterar v0.6)

In [ ]:
def save_csv(path,items,fields=None):
    if fields is None: fields=list(items[0]) if items else ['empty']
    with path.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore')
        w.writeheader()
        w.writerows(items)
summary={
 'audit_version':'0.7','timestamp_utc':STAMP,'source_path':str(SRC),
 'note':'Audit only. No rights clearance, no corpus inclusion, no tokenizer counts.',
 'languages':stats,'manifest_issues':manifest_issues,
 'repeated_id_with_different_hash':[
     {'language':k[0],'book_id':k[1],'hashes':v} for k,v in repeated_ids.items()],
 'duplicate_manifest_sha':[
     {'language':k[0],'sha256':k[1],'occurrences':v} for k,v in duplicate_manifest_sha.items()],
 'orphans':orphans,
 'integrity_failures':[
     {'language':r['language'],'book_id':r['book_id'],'sha256':r['sha256'],
      'exists':r['exists'],'sha_ok':r['sha_ok'],'bytes_ok':r['bytes_ok'],
      'characters_ok':r['characters_ok'],'read_error':r.get('read_error')}
     for r in failures],
 'rights_approved_count':0
}
report_path=OUT/f'audit_summary_{STAMP}.json'
inventory_path=OUT/f'rights_inventory_{STAMP}.csv'
records_path=OUT/f'audit_records_{STAMP}.csv'
report_path.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
save_csv(inventory_path,inventory)
save_csv(records_path,rows)
print('Relatório:',report_path)
print('Inventário:',inventory_path)
print('Auditoria por arquivo:',records_path)

## 8. Critérios de passagem à próxima fase
Antes de escalar a coleta: (a) zero falhas de integridade; (b) investigar IDs associados a hashes distintos; (c) registrar evidência de direitos por obra/edição/jurisdição; (d) definir limpeza de boilerplate **em cópias derivadas**; (e) deduplicar contra v8 e v9; (f) contar tokens com o MaiaTokenizer-65k definitivo.

**Importante:** esta auditoria não tenta determinar domínio público automaticamente. Textos íntegros continuam em quarentena.